In [1]:
from transformers import AutoConfig
import __init__

configs = {
    '1b': AutoConfig.from_pretrained('Qwen/Qwen2.5-1.5B-Instruct'),
    '1b-instruct': AutoConfig.from_pretrained('Qwen/Qwen2.5-1.5B-Instruct'),
    '7b': AutoConfig.from_pretrained('Qwen/Qwen2.5-7B-Instruct'),
    '7b-instruct': AutoConfig.from_pretrained('Qwen/Qwen2.5-7B-Instruct'),
    '14b': AutoConfig.from_pretrained('Qwen/Qwen2.5-14B-Instruct'),
    '14b-instruct': AutoConfig.from_pretrained('Qwen/Qwen2.5-14B-Instruct'),
    '32b': AutoConfig.from_pretrained('Qwen/Qwen2.5-32B-Instruct'),
    '32b-instruct': AutoConfig.from_pretrained('Qwen/Qwen2.5-32B-Instruct')
}

def many_decode_flops(p_l, s, h, h_, n):
    # return 8*s*(h^2) + 16*s*h + 4*s*p_l*h + 4*s*p_l*n + 6*s*h*h_ + 2*s*h_
    # return cum_decode_flop(p_l, s, h, h_, n)/5
    return decode_flop(p_l, h, h_, n)


def prefill_flop(s, h, h_, n):
    return 8*s*(h^2) + 16*s*h + 4*(s^2)*h + 4*(s^2)*n + 6*s*h*h_ + 2*s*h_

def decode_flop(s, h, h_, n):
    return 8*(h^2) + 16*h + 4*s*h + 4*s*n + 6*h*h_ + 2*h_

def cum_decode_flop(p_l, l, h, h_, n):
    flops = 0
    for i in range(l):
        flops = flops + decode_flop(p_l+i, h, h_, n)
    return flops

def get_flops(name, help_model, token_usages):
    prefill_flops, prefix_flops, decode_flops=0,0,0
    spe_config = configs[name]
    tgt_config = configs[help_model]
    spe_info = spe_config.hidden_size, spe_config.intermediate_size, spe_config.num_attention_heads
    tgt_info = tgt_config.hidden_size, tgt_config.intermediate_size, tgt_config.num_attention_heads
        
    accept_tokens = token_usages['correct_tokens']
    completion_tokens = token_usages['completion_tokens']
    prompt_tokens = token_usages['prompt_tokens']
    prefix_n, prompt_len = len(accept_tokens[1:]), prompt_tokens

    prefill_flops = prefill_flops+prefill_flop(prompt_tokens, *spe_info)
    prefill_flops = prefill_flops+prefill_flop(prompt_tokens, *tgt_info)
    print(completion_tokens, accept_tokens[0]*prefix_n)
    decode_flops = cum_decode_flop(prompt_tokens, completion_tokens, *spe_info)
    for t in range(accept_tokens[0]):
        f = many_decode_flops(prompt_len, prefix_n, *tgt_info)
        prefix_flops = prefix_flops+f
        prompt_len = prompt_len+prefix_n
    # print(prompt_len, completion_tokens)
    return prefill_flops, prefix_flops, decode_flops, sum(accept_tokens[1:])


In [2]:
keywords = {"wait", "hmm", "alternatively", 'double-check'}
import re
def segement(text):
    text = text.strip()  # 处理原始文本

    # **按 `\n\n` 进行初步分割**
    sections = re.split(r'\n\n+', text)

    # **存储最终分割的部分**
    filtered_sections = []
    temp_section = ""

    for section in sections:
        last_20_chars = section[:30].lower()  # 取后 20 个字符并转换为小写
        # print(last_20_chars)
        if any(keyword in last_20_chars for keyword in keywords):
            # 如果当前 `temp_section` 不是空的，则合并存储
            if temp_section:
                filtered_sections.append(temp_section.strip())
            temp_section = section+ "\n\n"
        else:
            # 累积无关键词的部分，等待下次匹配
            temp_section += section + "\n\n"

    # **如果最后 `temp_section` 仍有内容，加入最终结果**
    if temp_section:
        filtered_sections.append(temp_section.strip())
    return len(filtered_sections)

In [3]:
import os
import json
import math
import re

def log_flops(flops):
    # print(flops)
    if flops <= 0: return 0
    r = 312*10**8
    # print(flops/r)
    return flops/r #

def count_exact_wait_in_code(code_str):
    # count = code_str.lower().count('wait')
    count = segement(code_str.lower())
    # print(count)
    # matches = re.findall(r'\bwait\b', code_str.lower(), flags=re.IGNORECASE)
    # # print(f"'wait' 精确出现次数: {len(matches)}")
    # count = len(matches)
    return count

def comput_acceptance_counts(accept_tokens):
    acceptance_counts = [0] * len(accept_tokens[0])
    for output in accept_tokens:
        for step, count in enumerate(output):
            acceptance_counts[step] += count
    rate = sum(acceptance_counts[1:]) / (acceptance_counts[0]*len(accept_tokens[0][1:]))
    print(acceptance_counts)
    # print(f"mean acceptance length: {sum(acceptance_counts[1:]) / (acceptance_counts[0]*len(accept_tokens[0][1:])):.2f}")
    return rate, acceptance_counts[0], sum(acceptance_counts[1:])

def get_info(data, name, help_model, t=0.6):
    print(name, help_model)
    correct_num, correct_tokens, worng_tokens, all_time_spend = 0, 0, 0, 0
    fix_tokens, prefill_flops, prefix_flops, decode_flops, = 0, 0, 0, 0
    accept_tokens = []
    right_wait_count,wrong_wait_count = 0,0
    for k in data.keys():
        response = data[k]['responses'][str(t)][0]['content']
        wait_count = count_exact_wait_in_code(response)
        token_usages = data[k]['token_usages'][str(t)][0]
        accept_tokens.append(token_usages['correct_tokens'])
        tokens_num = token_usages['completion_tokens']
        # print(token_usages['correct_tokens'], tokens_num,)
        prefill_flop, prefix_flop, decode_flop, fix_token = get_flops(name, help_model, token_usages)
        prefill_flops = prefill_flops + prefill_flop
        prefix_flops = prefix_flops + prefix_flop
        decode_flops = decode_flops + decode_flop
        if data[k]['responses'][str(t)][0]['correctness']: 
            correct_num = correct_num+1
            correct_tokens = correct_tokens+tokens_num
            right_wait_count = right_wait_count + wait_count
        else:
            worng_tokens = worng_tokens+tokens_num
            wrong_wait_count = wrong_wait_count + wait_count
        #  all_time_spend = all_time_spend + time_spend
    print(comput_acceptance_counts(accept_tokens))
    all_num = len(data.keys())
    return {
        'right_wait_count': right_wait_count,
        'wrong_wait_count': wrong_wait_count,
        'all_num':all_num,
        'correct_num':correct_num,
        'wrong_num':all_num-correct_num,
        'acc':round(correct_num/all_num, 4),
        'avg_token_num': round((correct_tokens+worng_tokens)/all_num, 2),
        'correct_avg_tokens':round(correct_tokens/correct_num, 2) if correct_num != 0 else 0,
        'worng_avg_tokens':round(worng_tokens/(all_num-correct_num), 2) if all_num-correct_num != 0 else 0,
        'prefill_flops': round(log_flops(prefill_flops), 2),
        'prefix_flops': round(log_flops(prefix_flops), 2),
        'decode_flops': round(log_flops(decode_flops), 2),
        'all_tokens': correct_tokens+worng_tokens,
        't/flops': round((correct_tokens+worng_tokens)/log_flops(decode_flops+prefix_flops+prefill_flops), 2),
        'all_flops': round(log_flops(decode_flops+prefix_flops+prefill_flops), 2),
        # 'avg_time_spend': round((correct_tokens+worng_tokens)/all_time_spend, 2),
        'fix_tokens': comput_acceptance_counts(accept_tokens)[1],
        'fix_ratio': comput_acceptance_counts(accept_tokens)[0]
    }

In [4]:
from collections import defaultdict
results = defaultdict(dict)

aime_data = dict()
folder_path = "/home/wxy320/ondemand/program/speculative_thinking/eval/decode"  # 替换为你的文件夹路径
json_files = [f for f in os.listdir(folder_path) if f.endswith(".json")]
for file in json_files:
    name = file.split('_')[0].split('-')[-1].lower()
    dataset = file.split('_')[1]
    file_path = os.path.join(folder_path, file)
    with open(file_path, "r", encoding="utf-8") as f:
        data = json.load(f)
        result = get_info(data, name, '7b')
    if 'aime' in dataset:
        if name not in aime_data.keys():
            aime_data[name] = dict()
        aime_data[name].update(data)
    results[(name,dataset)] = {
        'normal': result
    }
for name, data in aime_data.items():
    result = get_info(data, name, '7b')
    results[(name,'aime')] = {
        'normal': result
    }

32b 7b
1639 1675
1276 1235
1626 1475
5165 4955
4638 4425
10983 10320
13443 12570
7005 7295
1958 1780
4605 4705
4417 4220
4013 3875
5839 5720
6917 6555
10670 10240
1194 1125
4179 4035
1971 1985
1322 1210
3382 3305
5250 4920
5254 5140
4564 4270
10086 10885
3495 3480
2055 1880
21834 22085
17097 17535
5885 5600
5148 5090
4559 4445
1502 1395
2453 2315
2687 2440
12885 14145
5051 5150
10402 10450
1738 1620
3577 3385
11374 11710
[46129, 42536, 39601, 37133, 34874, 32949]
(0.8111730148063041, 46129, 187093)
[46129, 42536, 39601, 37133, 34874, 32949]
[46129, 42536, 39601, 37133, 34874, 32949]
32b 7b
2548 2450
8172 7555
28154 27960
15992 17930
17596 16390
9658 9165
4817 4695
12203 12935
3710 3485
3219 3050
6148 5940
16851 17330
5144 4870
3881 3520
6248 6390
2680 2720
4901 4970
9077 8655
20576 20290
19057 19290
11553 11040
12190 11615
25296 23625
7814 8085
4541 4195
15226 17180
17447 19810
5908 5560
7205 6965
17573 19130
[65359, 59889, 55303, 51521, 48181, 45199]
(0.795890389999847, 65359, 260093)

In [5]:
final_results = defaultdict(dict)
for key, r in results.items():
    # print(key)
    final_results[key[1]][key[0]]=r
for d, r in final_results.items():
    if d == 'aime' or d == 'math500':
        print(d)
        print('------------------------')
        for k, r_ in r.items():
            print(k)
            for k_, r_1 in r_.items():
                print(k_, r_1['acc'],  r_1['t/flops'], r_1['fix_ratio'])
        print('------------------------')

math500
------------------------
32b
normal 0.936 29.5 0.8161130259586726
------------------------
aime
------------------------
32b
normal 0.6 27.0 0.8038668246837362
------------------------
